# RAG Agent (검색 증강 생성 에이전트)

**RAG (Retrieval-Augmented Generation)** 는 외부 지식 소스에서 정보를 검색하여 LLM의 응답을 향상시키는 기술입니다.

RAG 에이전트는 검색 도구를 사용하여 관련 문서를 찾고, 이를 컨텍스트로 활용하여 답변을 생성합니다.

**참고**: [LangChain 공식 문서 - RAG Agent](https://docs.langchain.com/oss/python/langchain/rag)

## RAG 워크플로우

1. **인덱싱 (Indexing)**: 문서를 로드, 분할, 임베딩하여 벡터 스토어에 저장
2. **검색 및 생성 (Retrieval & Generation)**: 사용자 질문에 대해 관련 문서를 검색하고 답변 생성

## RAG 패턴 이해

RAG는 두 가지 주요 접근 방식이 있습니다.
이 노트북에서는 **에이전트 기반 RAG를 중심으로** 다루고, 체인 기반 RAG는 마지막에 참고로 비교합니다.

### 패턴 1: 에이전트 기반 RAG (Agentic RAG)

에이전트가 필요할 때만 검색 도구를 호출합니다.
- 장점: 유연성, 필요할 때만 검색
- 단점: 여러 번의 모델 호출로 인한 지연 시간

### 패턴 2: 체인 기반 RAG (Chain-based RAG)

항상 검색을 먼저 실행하고 결과를 컨텍스트로 제공합니다.
- 장점: 단일 모델 호출로 빠름
- 단점: 항상 검색하므로 유연성 낮음

## 학습 내용
- RAG Agent를 위한 벡터 스토어 구축 (로드 → 분할 → 임베딩 → 저장)
- 검색 도구 정의 및 에이전트 통합
- 에이전트 기반 RAG vs 체인 기반 RAG

In [ ]:
# 모델 및 임베딩 초기화
# 벡터 스토어 생성 (간단한 실습용 인메모리 방식)
# 영구 저장이 필요하면 Chroma 등으로 교체 가능:
#   from langchain_chroma import Chroma
#   vector_store = Chroma(collection_name="nike_10k", embedding_function=embeddings,
#                         persist_directory="./chroma_langchain_db")

## 1. 인덱싱: RAG Agent를 위한 벡터 스토어 구축

에이전트가 검색할 지식 베이스를 만드는 단계입니다.
PDF 문서를 로드하고, 검색에 적합한 크기로 분할한 뒤, 임베딩하여 벡터 스토어에 저장합니다.

LangChain은 다양한 문서 소스를 지원합니다: PDF, 웹페이지, 데이터베이스, API 등.
(웹페이지는 `WebBaseLoader`, PDF는 `PyPDFLoader` 사용)

In [ ]:
# pip install pypdf

### 1.1 문서 로드

In [ ]:
# PDF 파일 로드 (나이키 10-K 연례 보고서, 한국어 번역본)

### 1.2 텍스트 분할 (Text Splitting)

페이지 단위는 검색에 너무 큽니다. 더 작은 청크로 분할하여 정확한 검색이 가능하도록 합니다.

**중요 설정:**
- `chunk_size`: 각 청크의 최대 크기
- `chunk_overlap`: 청크 간 중첩 (문맥 유지)

In [ ]:
# 텍스트 분할기 설정
# 문서 분할

### 1.3 임베딩 후 벡터 스토어에 저장

텍스트를 숫자 벡터로 변환하여 저장합니다. 의미가 유사한 텍스트는 벡터 공간에서 가까운 위치에 있으므로,
질문과 의미가 비슷한 청크를 유사도 기반으로 찾을 수 있습니다.

In [ ]:
# 문서를 벡터 스토어에 추가 (내부적으로 임베딩 수행)

### 1.4 검색 동작 확인

에이전트에 연결하기 전에 벡터 스토어가 관련 문서를 잘 찾는지 확인합니다.

In [ ]:
# 유사도 검색 테스트

## 2. 에이전트 기반 RAG (Agentic RAG)

벡터 스토어 검색을 **도구(tool)** 로 감싸서 에이전트에 제공합니다.
에이전트는 질문에 답하기 위해 필요하다고 판단할 때 검색 도구를 호출합니다.

### 2.1 검색 도구 생성

`response_format="content_and_artifact"` 를 사용하면 LLM에게 전달할 문자열(content)과
원본 문서 객체(artifact)를 함께 반환할 수 있습니다.

In [ ]:
def retrieve_context(query: str):
    # 벡터 스토어에서 유사한 문서 검색
    # 검색된 문서를 문자열로 직렬화
    # 문자열과 문서 객체를 함께 반환
# 도구 테스트

### 2.2 RAG 에이전트 생성

In [ ]:
# 시스템 프롬프트 설정
# 에이전트 생성

### 2.3 에이전트 실행

스트리밍 방식으로 실행하면 에이전트의 동작 과정(도구 호출 → 검색 결과 → 최종 답변)을 단계별로 볼 수 있습니다.

In [ ]:
# 질문에 대한 답변 생성
# 스트리밍 방식으로 실행

### 2.4 여러 질문 테스트

## 3. 참고 : 체인 기반 RAG (Chain-based RAG) 

에이전트가 판단하지 않고 **항상 검색을 먼저 실행**한 뒤 결과를 컨텍스트로 제공하는 방식입니다.
단일 모델 호출이므로 빠르지만, 검색이 불필요한 질문에도 항상 검색을 수행합니다.

In [ ]:
# 체인 기반 RAG 예제
def chain_based_rag(query: str):
    # 1. 검색
    # 2. 컨텍스트와 함께 답변 생성
# 체인 기반 RAG 테스트

### 실습 문제

본문에서 배운 **인덱싱 → 검색 도구 → RAG 에이전트** 전체 흐름을 새로운 문서에 적용해 **블로그 QA 에이전트**를 완성하세요.

1. **인덱싱**:
   `WebBaseLoader`로 웹 문서 https://botpress.com/ko/blog/llm-agents 를 로드하고,
   `RecursiveCharacterTextSplitter`(chunk_size=1000, chunk_overlap=200)로 분할한 뒤,
   본문의 나이키 벡터 스토어와는 **별도의 새 `InMemoryVectorStore`** 에 저장하세요.

2. **검색 도구**:
   `@tool(response_format="content_and_artifact")` 데코레이터로
   `retrieve_blog_context(query)` 도구를 정의하세요.
   벡터 스토어에서 `k=3`으로 검색하고, 출처(source)를 포함한 문자열로 직렬화해 반환하세요.

3. **RAG 에이전트**:
   `create_agent`로 에이전트를 생성하세요. system prompt에
   **"반드시 검색된 정보에 근거해서 답변하고, 검색 결과에 없는 내용은 모른다고 답하세요"** 지침을 포함하세요.

4. **테스트**: 아래 질문들로 에이전트를 실행하고, 도구 호출 여부와 답변을 확인하세요.

### 테스트 입력 예시

* `"LLM 에이전트를 정의하는 4가지 특징은 무엇인가요?"`
  👉 retrieve_blog_context 도구 호출 → 블로그 내용에 근거한 답변

* `"나이키의 2023년 매출은 얼마였나요?"`
  👉 블로그에 없는 내용 → 에이전트가 모른다고 답하는지(환각 방지) 확인